In [ ]:
# 检查dimer中的所有pdb的L链是否包含C，删除包含的pdb文件
import os
from Bio.PDB import PDBParser

def check_chain_c(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file)
    
    for model in structure:
        for chain in model:
            if chain.id == 'L':
                for residue in chain:
                    if residue.get_resname() == 'CYS':  # 检查是否包含CYS残基
                        return True
    return False

pdb_dir = 'dimer'  # 替换为你的pdb文件夹路径
os.makedirs('dimer_noC', exist_ok=True)

with open(f"{pdb_dir}/PDB_noCYS.list", 'w') as f:
    for pdb_file in os.listdir(pdb_dir):
        if pdb_file.endswith('.pdb'):
            full_path = os.path.join(pdb_dir, pdb_file)
            if check_chain_c(full_path) == False:
                # print(f"Moving {pdb_file} because it doesn't contain CYS in chain L.")
                f.write(pdb_file + '\n')

In [2]:
# 检查 PepSet-dimer_noC 的所有 PDB：过滤掉有断链的结构，并保存处理后的结构
import os
from Bio.PDB import PDBParser, PDBIO

parser = PDBParser(QUIET=True)
input_dir = "PepSet-dimer_noC"
output_dir = "cleaned_allchains"
os.makedirs(output_dir, exist_ok=True)

total_pdb = 0
skipped_discontinuous = 0
saved_count = 0

if not os.path.isdir(input_dir):
    print(f"[ERROR] 输入目录不存在: {input_dir}")
else:
    for filename in os.listdir(input_dir):
        if not filename.endswith(".pdb"):
            continue

        total_pdb += 1
        pdb_path = os.path.join(input_dir, filename)
        structure = parser.get_structure(filename, pdb_path)

        # 1) 检查每条链是否有断裂
        all_chains_continuous = True
        for model in structure:
            for chain in model:
                resseq_list = [res.id[1] for res in chain if res.id[0] == " "]
                if len(resseq_list) <= 1:
                    continue
                resseq_list = sorted(set(resseq_list))
                if any((b - a) > 1 for a, b in zip(resseq_list[:-1], resseq_list[1:])):
                    all_chains_continuous = False
                    break
            if not all_chains_continuous:
                break

        if not all_chains_continuous:
            skipped_discontinuous += 1
            continue

        # 2) 非 L 链改名为 A 链，并将每条链的标准残基重新编号为 1..N
        for model in structure:
            for chain in model:
                if chain.id != "L":
                    chain.id = "A"

                new_resseq = 1
                for res in chain:
                    if res.id[0] == " ":  # 仅标准残基
                        res.id = (" ", new_resseq, " ")
                        new_resseq += 1

        # 3) 保存到新目录（保留原文件）
        io = PDBIO()
        io.set_structure(structure)
        io.save(os.path.join(output_dir, filename))
        saved_count += 1

    print(f"总 PDB 文件数: {total_pdb}")
    print(f"因断链跳过: {skipped_discontinuous}")
    print(f"成功保存: {saved_count}")
    print(f"输出目录: {output_dir}")

/home/junjiechen/miniconda3/envs/Dpepalign/lib/python3.13/site-packages/Bio/PDB/Entity.py:197: BiopythonWarning: The id `(' ', 1, ' ')` is already used for a sibling of this entity. Changing id from `(' ', 0, ' ')` to `(' ', 1, ' ')` might create access inconsistencies to children of the parent entity.
  warnings.warn(
/home/junjiechen/miniconda3/envs/Dpepalign/lib/python3.13/site-packages/Bio/PDB/Entity.py:197: BiopythonWarning: The id `(' ', 2, ' ')` is already used for a sibling of this entity. Changing id from `(' ', 1, ' ')` to `(' ', 2, ' ')` might create access inconsistencies to children of the parent entity.
  warnings.warn(
/home/junjiechen/miniconda3/envs/Dpepalign/lib/python3.13/site-packages/Bio/PDB/Entity.py:197: BiopythonWarning: The id `(' ', 3, ' ')` is already used for a sibling of this entity. Changing id from `(' ', 2, ' ')` to `(' ', 3, ' ')` might create access inconsistencies to children of the parent entity.
  warnings.warn(
/home/junjiechen/miniconda3/envs/Dpep

总 PDB 文件数: 157
因断链跳过: 39
成功保存: 118
输出目录: cleaned_allchains


In [4]:
# 检查dimer_filtered中的pdb有多少是属于/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold/PepSet_dimer_noC.list中的
import os
import pandas as pd
pep_filter_dir = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/dimer_filtered/results-pep_plddt.csv'
list_file = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/dimer_noCys_filtered/PepSet_dimer_noC.list'
with open(list_file, 'r') as f:
    valid_pdbs = set(line.strip() for line in f)
count = 0

df = pd.read_csv(pep_filter_dir, sep=',')
pdbs = df["pdb"].to_list()

for pdb in pdbs:
      if pdb in valid_pdbs:
        count += 1
print(f"有 {count} 个 PDB 文件属于 {list_file} 中的列表。")

有 83 个 PDB 文件属于 /home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/dimer_noCys_filtered/PepSet_dimer_noC.list 中的列表。
